# Data preparation and cleaning

## 1) Load all data of accepted loans

[Source](https://www.kaggle.com/datasets/wordsforthewise/lending-club/data)
[Feature explanation](https://github.com/bacover/Capstone-LendingClub/blob/main/LCDataDictionary.xlsx) or in `/data/Features description LendingClub.xlsx`


In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

# Suppress DtypeWarning when reading the CSV file
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# Load the dataset
ORIGINAL_DATA_PATH = Path("data/accepted_2007_to_2018Q4.csv")
data = pd.read_csv(ORIGINAL_DATA_PATH)

## 2) Set columns for prediction
Using only finished statuses of loans.
For classification converted `Fully Paid` to 0 and `Charged Off` or `Default` to 1.

In [2]:
# Display the count of each unique value in the "loan_status" column
data["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [3]:
# Create a binary target variable based on the "loan_status" column
# and drop the original "loan_status" column
print("Original length of data:", len(data))

valid_statuses = ["Fully Paid", "Charged Off", "Default"]
data = data[data["loan_status"].isin(valid_statuses)].copy()
data["target"] = np.where(data["loan_status"] == "Fully Paid", 0, 1)
data = data.drop(columns=["loan_status"])

print("Length of data after filtering valid loan statuses:", len(data))

Original length of data: 2260701
Length of data after filtering valid loan statuses: 1345350


Calculate the ROI as regression target

$$\texttt{target\_annual\_roi} = \frac{\texttt{total\_pymnt}}{\texttt{loan\_amnt}}^\frac{1}{\texttt{actual\_duration\_years}} - 1$$

In [4]:
# Drop records with missing values in the "last_pymnt_d", "total_pymnt", "loan_amnt"
# or "issue_d" columns to be able to calculate "target_annual_roi"
print(
    "Dropped rows with missing 'last_pymnt_d', 'total_pymnt','loan_amnt', or 'issue_d' values:"
)
print(data[["last_pymnt_d", "total_pymnt", "loan_amnt", "issue_d"]].isna().sum())

data = data.dropna(subset=["last_pymnt_d", "total_pymnt", "loan_amnt", "issue_d"])

print("Remaining rows after dropping:", len(data))

Dropped rows with missing 'last_pymnt_d', 'total_pymnt','loan_amnt', or 'issue_d' values:
last_pymnt_d    2313
total_pymnt        0
loan_amnt          0
issue_d            0
dtype: int64
Remaining rows after dropping: 1343037


In [5]:
# Calculate the target variable "target_annual_roi" based on the actual
# duration of the loan in years

# Convert "issue_d" and "last_pymnt_d" to datetime format
data["issue_d"] = pd.to_datetime(data["issue_d"], format="%b-%Y")
data["last_pymnt_d"] = pd.to_datetime(data["last_pymnt_d"], format="%b-%Y")

# Calculate the actual duration of the loan in months
data["actual_duration_months"] = (
    data["last_pymnt_d"].dt.year - data["issue_d"].dt.year
) * 12 + (data["last_pymnt_d"].dt.month - data["issue_d"].dt.month)

# Replace any zero durations with 1 month to avoid division by zero
data["actual_duration_months"] = data["actual_duration_months"].replace(0, 1)

actual_years = data["actual_duration_months"] / 12

data["target_annual_roi"] = (data["total_pymnt"] / data["loan_amnt"]) ** (
    1 / actual_years) - 1

In [6]:
# Check for any remaining missing values in the "target_annual_roi" column
print("Missing values in 'target_annual_roi':", data["target_annual_roi"].isna().sum())

Missing values in 'target_annual_roi': 0


## 3) Chronological train test split

In [ ]:
# Use the "id" column as the index of the DataFrame
data = data.set_index("id")

# Sort the data by "issue_d"
data = data.sort_values("issue_d")

# Split the data into training and testing sets based on the "issue_d" column
split_idx = int(len(data) * 0.8)
train_df = data.iloc[:split_idx]
test_df = data.iloc[split_idx:]

print("Training set size:", len(train_df))
print("Testing set size:", len(test_df))
print("Training set date range:", train_df["issue_d"].min(), train_df["issue_d"].max())
print("Testing set date range:", test_df["issue_d"].min(), test_df["issue_d"].max())

Training set size: 1074429
Testing set size: 268608
Training set date range: 2007-06-01 00:00:00 2016-10-01 00:00:00
Testing set date range: 2016-10-01 00:00:00 2018-12-01 00:00:00


## 4) Find columns to drop

In [8]:
# Check for differences between "loan_amnt" and "funded_amnt" in the training set
# Nearly all records should have "loan_amnt" equal to "funded_amnt"
train_df[
    (train_df["loan_amnt"] != train_df["funded_amnt"])
    & (train_df["loan_amnt"] > 0)
    & (train_df["funded_amnt"] > 0)
]

,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,...,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,target,actual_duration_months,target_annual_roi
id,,,,,,,,,,,,,,,,,,,,,
140036,NaN,4000.0,1000.0,550.000000,36 months,9.96,32.25,B,B5,Schering-Plough Corp.,...,N,NaN,NaN,NaN,NaN,NaN,NaN,1,27,-0.492187
139980,NaN,12000.0,5000.0,1399.996684,36 months,10.91,163.49,C,C3,Staples,...,N,NaN,NaN,NaN,NaN,NaN,NaN,1,29,-0.314330
141774,NaN,20000.0,4800.0,1949.999594,36 months,13.12,162.01,D,D5,Legal Services of New Jersey,...,N,NaN,NaN,NaN,NaN,NaN,NaN,1,10,-0.951535
137042,NaN,25000.0,5650.0,725.000000,36 months,14.38,194.15,E,E4,MSIP,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0,36,-0.346120
180712,NaN,22550.0,12000.0,2225.000000,36 months,15.96,421.65,F,F4,Polygon Company,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0,36,-0.123602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5335336,NaN,27500.0,27475.0,27475.000000,60 months,23.83,787.70,F,F5,Internal Revenue Service,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0,21,0.203396
5345475,NaN,25000.0,24975.0,24975.000000,60 months,23.83,716.02,F,F5,U.S Army Materiel Command,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0,3,0.253545
5649239,NaN,33425.0,33325.0,33275.000000,60 months,19.72,877.73,D,D5,Dunbar Armored,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0,60,0.095185


In [9]:
# Check the consistency of "grade" and "sub_grade" columns in the training set
print(
    "Consistent rows of 'grade' and 'sub_grade':",
    sum((train_df["grade"] == train_df["sub_grade"].str[0])),
    "\nTotal rows:",
    len(train_df),
)

Consistent rows of 'grade' and 'sub_grade': 1074429 
Total rows: 1074429


Also used Data Wrangler VSCode Extension

In [10]:
# Columns to drop
cols_to_drop = [
    "member_id",  # 100% NA
    "funded_amnt",  # 99,9% same as loan_amnt
    "funded_amnt_inv",  # Not known at the time of application
    "grade",  # 100% values correspond to letter in sub_grade
    "emp_title",  # 31% different values - text values - would require text mining
    "pymnt_plan",  # 100% values "n"
    "url",  # 100% unique not useful values
    "desc",  # Text mining needed
    "title",  # Text mining needed
    "zip_code",  # 939 categories and corresponding to addr_state
    "out_prncp",  # Info about target
    "out_prncp_inv",  # Info about target
    "total_pymnt",  # Info about target
    "total_pymnt_inv",  # Info about target
    "total_rec_prncp",  # Info about target
    "total_rec_int",  # Info about target
    "total_rec_late_fee",  # Info about target
    "recoveries",  # Info about target
    "collection_recovery_fee",  # Info about target
    "last_pymnt_d",  # Info about target
    "last_pymnt_amnt",  # Info about target
    "next_pymnt_d",  # Info about target
    "last_credit_pull_d",  # Info about target
    "last_fico_range_high",  # Info about target
    "last_fico_range_low",  # Info about target
    "policy_code",  # All values are 1
    "revol_bal_joint",  # 100% NA in train set
    "sec_app_fico_range_low",  # 100% NA in train set
    "sec_app_fico_range_high",  # 100% NA in train set
    "sec_app_earliest_cr_line",  # 100% NA in train set
    "sec_app_inq_last_6mths",  # 100% NA in train set
    "sec_app_mort_acc",  # 100% NA in train set
    "sec_app_open_acc",  # 100% NA in train set
    "sec_app_revol_util",  # 100% NA in train set
    "sec_app_open_act_il",  # 100% NA in train set
    "sec_app_num_rev_accts",  # 100% NA in train set
    "sec_app_chargeoff_within_12_mths",  # 100% NA in train set
    "sec_app_collections_12_mths_ex_med",  # 100% NA in train set
    "sec_app_mths_since_last_major_derog",  # 100% NA in train set
    "hardship_flag",  # Info about target and 100% values "N"
    "hardship_type",  # Info about target
    "hardship_reason",  # Info about target
    "hardship_status",  # Info about target
    "deferral_term",  # Info about target
    "hardship_amount",  # Info about target
    "hardship_start_date",  # Info about target
    "hardship_end_date",  # Info about target
    "payment_plan_start_date",  # Info about target
    "hardship_length",  # Info about target
    "hardship_dpd",  # Info about target
    "hardship_loan_status",  # Info about target
    "orig_projected_additional_accrued_interest",  # Info about target
    "hardship_payoff_balance_amount",  # Info about target
    "hardship_last_payment_amount",  # Info about target
    "debt_settlement_flag",  # Info about target
    "debt_settlement_flag_date",  # Info about target
    "settlement_status",  # Info about target
    "settlement_date",  # Info about target
    "settlement_amount",  # Info about target
    "settlement_percentage",  # Info about target
    "settlement_term",  # Info about target
    "actual_duration_months",  # Not known at the time of application
]

In [11]:
# Columns to treat as unordered categorical variables
cols_to_unordered_cat = [
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state",
    "initial_list_status",
    "application_type",
    "disbursement_method",
]

In [12]:
# Columns to treat as ordered categorical variables
cols_to_ordered_cat = [
    "term",
    "sub_grade",
    "emp_length",
]

In [13]:
# Columns to engineer
cols_to_engineer = [
    "issue_d",  # Month and year, keep as datetime
    "earliest_cr_line",  # Month and year, difference with issue_d and keep as datetime
    "fico_range_low",  # Mean and drop
    "fico_range_high",  # Mean and drop
]

In [14]:
# Columns to combine from joint application
cols_to_combine_joint = ["annual_inc_joint", "dti_joint", "verification_status_joint"]

## 5) Columns dropping and adjustments

In [15]:
# Drop the columns
data = data.drop(columns=cols_to_drop)

Average fico - fico_range_low and fico_range_high has allways difference of 4 and in some rare cases 5

In [16]:
# Calculate the average FICO score and drop the original
# "fico_range_low" and "fico_range_high" columns
data["fico_avg"] = (data["fico_range_low"] + data["fico_range_high"]) / 2

# Display all three columns to verify the calculation of "fico_avg"
display(data[["fico_avg", "fico_range_low", "fico_range_high"]])

data = data.drop(columns = ["fico_range_low", "fico_range_high"])

,fico_avg,fico_range_low,fico_range_high
id,,,
87023,662.0,660.0,664.0
112323,682.0,680.0,684.0
99009,792.0,790.0,794.0
112245,772.0,770.0,774.0
109355,662.0,660.0,664.0
...,...,...,...
144587705,687.0,685.0,689.0
144359153,732.0,730.0,734.0
144793741,702.0,700.0,704.0


Datetime columns feature engineering

In [ ]:
# Convert "earliest_cr_line" to datetime format
data["earliest_cr_line"] = pd.to_datetime(
    data["earliest_cr_line"], format="%b-%Y", errors="coerce"
)

# Extract month and year from "issue_d" and "earliest_cr_line"
# Not using the year of "issue_d" due to irrelevant information
# for the models due to the time-based split of the data
data["issue_d_month"] = data["issue_d"].dt.month
data["earliest_cr_line_year"] = data["earliest_cr_line"].dt.year
data["earliest_cr_line_month"] = data["earliest_cr_line"].dt.month

# Calculate the length of credit history in months
# as the difference between "issue_d" and "earliest_cr_line"
data["credit_history_length_mths"] = (
    data["issue_d"].dt.year - data["earliest_cr_line_year"]
) * 12 + (data["issue_d"].dt.month - data["earliest_cr_line_month"])

# Convert "earliest_cr_line" to a numeric variable representing the
# number of days since January 1, 1970
data["earliest_cr_line_numeric"] = (
    data["earliest_cr_line"] - pd.Timestamp("1970-01-01")
).dt.days

# Calculate the sine and cosine transformations for the cyclical
# features to capture their cyclical nature for the models
cyclical_cols = ["issue_d_month", "earliest_cr_line_month"]
for col in cyclical_cols:
    if col in data.columns:
        data[f"{col}_sin"] = np.sin(2 * np.pi * data[col] / 12.0)
        data[f"{col}_cos"] = np.cos(2 * np.pi * data[col] / 12.0)

Joint applications engineering

In [18]:
# Use the joint values for "annual_inc", "dti" and "verification_status"
# for the records with "application_type" equal to "Joint App"
data["annual_inc"] = np.where(
    data["application_type"] == "Joint App",
    data["annual_inc_joint"],
    data["annual_inc"],
)
data["dti"] = np.where(
    data["application_type"] == "Joint App", data["dti_joint"], data["dti"]
)
data["verification_status"] = np.where(
    data["application_type"] == "Joint App",
    data["verification_status_joint"],
    data["verification_status"],
)

data = data.drop(columns=["annual_inc_joint", "dti_joint", "verification_status_joint"])

Conversion to category type

In [19]:
# Convert the categorical columns to the "category" data type
data[cols_to_unordered_cat] = data[cols_to_unordered_cat].astype("category")
data[cols_to_ordered_cat] = data[cols_to_ordered_cat].astype("category")

In [20]:
# Check the remaining object columns that were not converted to categorical data type
data.select_dtypes(include=['object']).columns

Index([], dtype='object')

Update train and test sets after adjustments on whole data

In [ ]:
split_idx = int(len(data) * 0.8)
train_df = data.iloc[:split_idx]
test_df = data.iloc[split_idx:]

print("Training set size:", len(train_df))
print("Testing set size:", len(test_df))
print("Training set date range:", train_df["issue_d"].min(), train_df["issue_d"].max())
print("Testing set date range:", test_df["issue_d"].min(), test_df["issue_d"].max())

Training set size: 1074429
Testing set size: 268608
Training set date range: 2007-06-01 00:00:00 2016-10-01 00:00:00
Testing set date range: 2016-10-01 00:00:00 2018-12-01 00:00:00


## 6) Work with categorical features

In [22]:
# Check the number of unique values in the remaining categorical columns in the training set
all_cat_cols = train_df.select_dtypes(include=["object", "category"]).columns

cat_counts = train_df[all_cat_cols].nunique().sort_values(ascending=False)

print("Unique values in categorical columns:")
print(cat_counts)

Unique values in categorical columns:
addr_state             51
sub_grade              35
purpose                14
emp_length             11
home_ownership          6
verification_status     3
term                    2
initial_list_status     2
application_type        2
disbursement_method     2
dtype: int64


In [23]:
# Check the distribution of the "disbursement_method" column in the training set
train_df["disbursement_method"].value_counts(normalize=True, dropna=False) * 100

disbursement_method
Cash         99.800918
DirectPay     0.199082
Name: proportion, dtype: float64

In [24]:
# Check the distribution of the "application_type" column in the training set
train_df["application_type"].value_counts(normalize=True, dropna=False) * 100

application_type
Individual    99.632735
Joint App      0.367265
Name: proportion, dtype: float64

In [25]:
# Check the distribution of the "initial_list_status" column in the training set
train_df["initial_list_status"].value_counts(normalize=True, dropna=False) * 100

initial_list_status
w    53.941303
f    46.058697
Name: proportion, dtype: float64

In [26]:
# Check the distribution of the "term" column in the training set
train_df["term"].value_counts(normalize=True, dropna=False) * 100

term
36 months    75.875465
60 months    24.124535
Name: proportion, dtype: float64

In [27]:
# Check the distribution of the "verification_status" column in the training set
train_df["verification_status"].value_counts(normalize=True, dropna=False) * 100

verification_status
Source Verified    38.331151
Verified           32.024359
Not Verified       29.644490
Name: proportion, dtype: float64

In [28]:
# Check the distribution of the "home_ownership" column in the training set
train_df["home_ownership"].value_counts(normalize=True, dropna=False) * 100

home_ownership
MORTGAGE    49.333832
RENT        40.250310
OWN         10.397337
OTHER        0.013402
NONE         0.004188
ANY          0.000931
Name: proportion, dtype: float64

In [29]:
# Check the distribution of the "emp_length" column in the training set
train_df["emp_length"].value_counts(normalize=True, dropna=False) * 100

emp_length
10+ years    32.802074
2 years       9.007947
3 years       7.964603
< 1 year      7.882792
1 year        6.564789
5 years       6.278591
4 years       5.954512
NaN           5.431908
8 years       4.844899
6 years       4.725766
7 years       4.649726
9 years       3.892393
Name: proportion, dtype: float64

In [30]:
# Check the distribution of the "purpose" column in the training set
train_df["purpose"].value_counts(normalize=True, dropna=False) * 100

purpose
debt_consolidation    58.629654
credit_card           22.630439
home_improvement       6.149778
other                  5.255443
major_purchase         2.067331
small_business         1.166666
car                    1.045672
medical                1.038598
moving                 0.664446
vacation               0.601436
house                  0.441630
wedding                0.212206
renewable_energy       0.066640
educational            0.030062
Name: proportion, dtype: float64

In [31]:
# Check the distribution of the "sub_grade" column in the training set
train_df["sub_grade"].value_counts(normalize=True, dropna=False) * 100

sub_grade
B4    6.371384
B3    6.348395
C1    6.216697
B5    5.955629
C2    5.868140
B2    5.655748
C3    5.519118
C4    5.372714
B1    5.189733
A5    4.951374
C5    4.655496
A4    4.014411
D1    3.885133
A1    3.278951
D2    3.238930
D3    2.788086
A3    2.749367
A2    2.712417
D4    2.677794
D5    2.221924
E1    1.893192
E2    1.692434
E3    1.438438
E4    1.198590
E5    0.996436
F1    0.773713
F2    0.577051
F3    0.477091
F4    0.376293
F5    0.293830
G1    0.205877
G2    0.155618
G3    0.111315
G4    0.076785
G5    0.061893
Name: proportion, dtype: float64

In [32]:
# Check the distribution of the "addr_state" column in the training set
train_df["addr_state"].value_counts(normalize=True, dropna=False) * 100

addr_state
CA    14.672724
NY     8.268857
TX     8.151493
FL     7.009211
IL     3.923014
NJ     3.683445
PA     3.415675
OH     3.311713
GA     3.250843
VA     2.867849
NC     2.790505
MI     2.602312
AZ     2.371864
MD     2.315369
MA     2.301408
WA     2.166732
CO     2.148583
MN     1.779457
MO     1.588099
IN     1.581491
TN     1.488605
CT     1.478087
NV     1.447001
WI     1.310557
AL     1.241590
OR     1.220183
SC     1.183047
LA     1.169458
KY     0.950551
OK     0.913788
KS     0.861853
AR     0.752679
UT     0.720848
NM     0.555458
HI     0.507432
NH     0.476718
MS     0.463130
RI     0.434370
WV     0.420595
MT     0.285547
DE     0.277822
DC     0.269166
AK     0.248039
WY     0.227284
NE     0.209227
SD     0.207459
VT     0.196942
ME     0.114572
ND     0.098471
ID     0.068222
IA     0.000652
Name: proportion, dtype: float64

Combine categories for home_ownership

In [33]:
# Replace the "home_ownership" categories "NONE" and "ANY"
# with "OTHER" due to their low frequency
home_cats_to_other = ["NONE", "ANY"]

data["home_ownership"] = (
    data["home_ownership"]
    .astype(object)
    .replace(home_cats_to_other, "OTHER")
    .astype("category")
)

# Check the unique values in the "home_ownership" column after the replacement
print(data["home_ownership"].unique())

['OWN', 'RENT', 'MORTGAGE', 'OTHER']
Categories (4, object): ['MORTGAGE', 'OTHER', 'OWN', 'RENT']


Convert ordered categories to integers

In [34]:
# Extract the number of months from the "term" column and convert it to an integer type
data["term_months"] = data["term"].astype(str).str.extract(r"(\d+)").astype("Int64")
data = data.drop(columns=["term"])

# Define the order of the categories for the "emp_length" and "sub_grade" columns
emp_length_order = [
    "< 1 year",
    "1 year",
    "2 years",
    "3 years",
    "4 years",
    "5 years",
    "6 years",
    "7 years",
    "8 years",
    "9 years",
    "10+ years",
]
sub_grade_order = sorted(data["sub_grade"].dropna().astype(str).unique())
sub_grade_mapping = {category: index for index, category in enumerate(sub_grade_order)}
emp_length_mapping = {
    category: index for index, category in enumerate(emp_length_order)
}

# Map the "sub_grade" and "emp_length" columns to their corresponding integer values
data["sub_grade"] = data["sub_grade"].astype(str).map(sub_grade_mapping).astype("Int64")
data["emp_length"] = (
    data["emp_length"].astype(str).map(emp_length_mapping).astype("Int64")
)

# Check the unique values in the "term_months", "sub_grade" and "emp_length"
# columns after the mapping
columns_to_check = ["term_months", "sub_grade", "emp_length"]
for col in columns_to_check:
    unique_vals = sorted(data[col].dropna().unique())
    print(f"Unique values in '{col}': {unique_vals}")

Unique values in 'term_months': [36, 60]
Unique values in 'sub_grade': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
Unique values in 'emp_length': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


Unification of numbers data type

In [35]:
# Check all integer type columns and convert them to "Int64"
old_int_cols = data.select_dtypes(include=["int32", "int64"]).columns
data[old_int_cols] = data[old_int_cols].astype("Int64")

# Check all float type columns and convert them to "float64"
float_cols = data.select_dtypes(include=["float32"]).columns
data[float_cols] = data[float_cols].astype("float64")

# Summarize the data types of the columns in the dataset after all transformations
print(data.dtypes.astype(str).value_counts())

float64           77
Int64              9
category           7
datetime64[ns]     2
Name: count, dtype: int64


Final train test split after all work

In [ ]:
split_idx = int(len(data) * 0.8)
train_df = data.iloc[:split_idx]
test_df = data.iloc[split_idx:]

print("Training set size:", len(train_df))
print("Testing set size:", len(test_df))
print("Training set date range:", train_df["issue_d"].min(), train_df["issue_d"].max())
print("Testing set date range:", test_df["issue_d"].min(), test_df["issue_d"].max())

Training set size: 1074429
Testing set size: 268608
Training set date range: 2007-06-01 00:00:00 2016-10-01 00:00:00
Testing set date range: 2016-10-01 00:00:00 2018-12-01 00:00:00


## 7) Save in parquet files for column types saving 

In [37]:
# Save the cleaned and preprocessed training and testing datasets to Parquet files
# to preserve the data types - have to convert the index to string type to avoid
# issues with saving and loading the Parquet files
TRAIN_DATA_PATH = Path("data/train_data.parquet")
TEST_DATA_PATH = Path("data/test_data.parquet")

train_df.index = train_df.index.astype(str)
test_df.index = test_df.index.astype(str)

train_df.to_parquet(TRAIN_DATA_PATH)
test_df.to_parquet(TEST_DATA_PATH)

train_df.index = pd.to_numeric(train_df.index)
test_df.index = pd.to_numeric(test_df.index)